# Local model and station archives

This starter notebook reads the operational meteo@uniparthenope archives that are mounted
**read-only** in every user server:

| Path | Content | Layout |
|---|---|---|
| `~/data/opendap` | Model output | `mmm/ddd/archive\|history/YYYY/MM/DD/mmm_ddd_YYYYMMDDZHHMM.nc` |
| `~/data/network` | Weather-station observations | `it.uniparthenope.meteo.wsN/YYYY/MM/DD/it.uniparthenope.meteo.wsN_YYYYMMDDZHH00.csv` |

Models (`mmm`): `wrf5` WRF, `rms3` ROMS, `ww33` WaveWatch III, `wcm3` WaComM, `aiq3` AIquam.
Domains (`ddd`): `d01`, `d02`, `d03`, … Product types: `archive` (normalized, ready for plotting)
and `history` (as produced by the model). Times are UTC.

The archives cannot be modified. Save derived products under `~/work`.

In [ ]:
import os
from datetime import datetime, timedelta, timezone
from pathlib import Path

import pandas as pd
import xarray as xr

OPENDAP = Path(os.environ.get("LAB_OPENDAP_DIR", "/home/jovyan/data/opendap"))
NETWORK = Path(os.environ.get("LAB_NETWORK_DIR", "/home/jovyan/data/network"))
OUTPUT = Path.home() / "work" / "outputs"


def model_file(model, domain, when, product="archive"):
    """Return the archive path of one model output time."""
    stamp = when.strftime("%Y%m%dZ%H%M")
    return OPENDAP / model / domain / product / when.strftime("%Y/%m/%d") / f"{model}_{domain}_{stamp}.nc"


def station_file(number, when):
    """Return the hourly CSV path of weather station wsN."""
    station = f"it.uniparthenope.meteo.ws{number}"
    return NETWORK / station / when.strftime("%Y/%m/%d") / f"{station}_{when.strftime('%Y%m%dZ%H')}00.csv"


print("model archive mounted:", OPENDAP.is_dir(), "| writable:", os.access(OPENDAP, os.W_OK))
print("station network mounted:", NETWORK.is_dir(), "| writable:", os.access(NETWORK, os.W_OK))

## Open a WRF output time

The most recent full hour from yesterday is used so that the file is complete.

In [ ]:
when = (datetime.now(timezone.utc) - timedelta(days=1)).replace(hour=12, minute=0, second=0, microsecond=0)
path = model_file("wrf5", "d03", when)
print(path)
if path.exists():
    with xr.open_dataset(path) as ds:  # read-only access
        display(ds)
else:
    print("File not available; list the directory to choose another time:")
    print(sorted(p.name for p in path.parent.glob("*.nc"))[:10] if path.parent.exists() else "directory missing")

## Read one hour of station observations

In [ ]:
csv = station_file(1, when)
print(csv)
if csv.exists():
    obs = pd.read_csv(csv, parse_dates=["Datetime"])
    display(obs[["Datetime", "name", "TempOut", "HumOut", "Barometer", "WindSpeed", "WindDir", "RainRate"]].head())
    ax = obs.set_index("Datetime")["TempOut"].plot(title=f"{obs['name'].iloc[0]} air temperature (°C)")
    ax.set_xlabel("UTC")
else:
    print("Observation file not available for this hour.")

## Save derived results in your workspace

Writing to `~/data` fails by design; write to `~/work` instead.

In [ ]:
OUTPUT.mkdir(parents=True, exist_ok=True)
try:
    (OPENDAP / "test-write").touch()
except OSError as exc:
    print("As expected, the archive is read-only:", exc.strerror)
print("Write derived products to", OUTPUT)